# Lucraftmeds — Tiền xử lý dữ liệu

Notebook này chuẩn hóa **một cột văn bản** trong CSV lớn mà không nạp toàn bộ file vào RAM.

Quy trình:

1. Chuẩn hóa Unicode về NFC.
2. Chuẩn hóa ký tự xuống dòng.
3. Loại bỏ whitespace ngang thừa nhưng giữ cấu trúc đoạn văn.
4. Ghi từng chunk sang một CSV mới; file nguồn không bị thay đổi.

> Trước khi chạy cell cuối, chỉ cần sửa `INPUT_CSV`, `OUTPUT_CSV` và `TEXT_COLUMN` trong cell cấu hình.

In [ ]:
from pathlib import Path
import os
import re
import tempfile
import time
import unicodedata

import pandas as pd


# --- Ba giá trị cần cập nhật trước khi chạy ---
INPUT_CSV = Path("path/to/input.csv")
OUTPUT_CSV = Path("path/to/preprocessed.csv")
TEXT_COLUMN = "TODO_TEXT_COLUMN"

# --- Cấu hình nâng cao (có thể giữ nguyên) ---
CHUNK_SIZE = 100_000
INPUT_ENCODING = "utf-8-sig"  # Đọc được cả UTF-8 thường và UTF-8 có BOM.
OUTPUT_ENCODING = "utf-8"
DELIMITER = ","
PREVIEW_ROWS = 5
OVERWRITE_OUTPUT = False

## Hàm chuẩn hóa văn bản

Whitespace ngang (space, tab, NBSP, …) được gộp thành một dấu cách. Xuống dòng được giữ lại và nhiều dòng trống liên tiếp được gộp thành tối đa một dòng trống.

In [ ]:
HORIZONTAL_WHITESPACE_RE = re.compile(r"[^\S\r\n]+")
EXCESS_BLANK_LINES_RE = re.compile(r"\n{3,}")


def normalize_text(value):
    """Chuẩn hóa Unicode và whitespace, đồng thời giữ nguyên giá trị null."""
    if value is None or value is pd.NA:
        return value

    try:
        if pd.isna(value):
            return value
    except (TypeError, ValueError):
        pass

    text = unicodedata.normalize("NFC", str(value))
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    cleaned_lines = [
        HORIZONTAL_WHITESPACE_RE.sub(" ", line).strip()
        for line in text.split("\n")
    ]
    text = "\n".join(cleaned_lines)
    text = EXCESS_BLANK_LINES_RE.sub("\n\n", text)
    return text.strip()


def preview_value(value, limit=200):
    """Tạo chuỗi xem trước ngắn gọn, không làm thay đổi dữ liệu ghi ra file."""
    if value is None or value is pd.NA:
        return "<NULL>"
    text = str(value).replace("\n", "\\n")
    return text if len(text) <= limit else text[: limit - 1] + "…"

## Pipeline xử lý CSV theo chunk

Dữ liệu được ghi vào file tạm trong cùng thư mục với đầu ra. Chỉ khi mọi chunk đã xử lý thành công, file tạm mới được đổi tên thành file kết quả. Nếu có lỗi, file tạm được xóa và ngoại lệ được báo lại.

In [ ]:
def validate_csv_job(
    input_csv,
    output_csv,
    text_column,
    *,
    input_encoding=INPUT_ENCODING,
    delimiter=DELIMITER,
    overwrite_output=OVERWRITE_OUTPUT,
):
    """Xác thực cấu hình và trả về đường dẫn tuyệt đối cùng danh sách cột."""
    if not text_column or text_column == "TODO_TEXT_COLUMN":
        raise ValueError("Hãy cập nhật TEXT_COLUMN bằng tên cột chứa văn bản.")

    input_path = Path(input_csv).expanduser().resolve()
    output_path = Path(output_csv).expanduser().resolve()

    if not input_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy CSV nguồn: {input_path}")
    if input_path == output_path:
        raise ValueError("INPUT_CSV và OUTPUT_CSV phải là hai file khác nhau.")
    if output_path.exists() and not overwrite_output:
        raise FileExistsError(
            f"File đầu ra đã tồn tại: {output_path}. "
            "Hãy chọn đường dẫn khác hoặc đặt OVERWRITE_OUTPUT = True."
        )

    header = pd.read_csv(
        input_path,
        sep=delimiter,
        encoding=input_encoding,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        nrows=0,
        on_bad_lines="error",
    )
    columns = header.columns.tolist()
    if text_column not in columns:
        raise KeyError(
            f"Không tìm thấy cột {text_column!r}. Các cột hiện có: {columns}"
        )

    return input_path, output_path, columns


def preprocess_csv(
    input_csv,
    output_csv,
    text_column,
    *,
    chunk_size=CHUNK_SIZE,
    input_encoding=INPUT_ENCODING,
    output_encoding=OUTPUT_ENCODING,
    delimiter=DELIMITER,
    preview_rows=PREVIEW_ROWS,
    overwrite_output=OVERWRITE_OUTPUT,
    show_progress=True,
):
    """Tiền xử lý cột text của CSV theo chunk và ghi kết quả nguyên tử."""
    if not isinstance(chunk_size, int) or chunk_size <= 0:
        raise ValueError("chunk_size phải là một số nguyên dương.")
    if not isinstance(preview_rows, int) or preview_rows < 0:
        raise ValueError("preview_rows phải là một số nguyên không âm.")

    input_path, output_path, columns = validate_csv_job(
        input_csv,
        output_csv,
        text_column,
        input_encoding=input_encoding,
        delimiter=delimiter,
        overwrite_output=overwrite_output,
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)

    file_descriptor, temporary_name = tempfile.mkstemp(
        prefix=f".{output_path.name}.",
        suffix=".tmp",
        dir=output_path.parent,
    )
    os.close(file_descriptor)
    temporary_path = Path(temporary_name)

    started_at = time.perf_counter()
    total_rows = 0
    changed_rows = 0
    empty_text_rows = 0
    chunk_count = 0
    samples = []

    try:
        # Ghi header trước để CSV rỗng vẫn tạo ra kết quả hợp lệ.
        pd.DataFrame(columns=columns).to_csv(
            temporary_path,
            sep=delimiter,
            encoding=output_encoding,
            index=False,
            lineterminator="\n",
        )

        reader = pd.read_csv(
            input_path,
            sep=delimiter,
            encoding=input_encoding,
            dtype=str,
            keep_default_na=False,
            na_filter=False,
            chunksize=chunk_size,
            on_bad_lines="error",
        )

        for chunk_count, chunk in enumerate(reader, start=1):
            original = chunk[text_column].copy()
            normalized = original.map(normalize_text)
            changed_mask = original.ne(normalized)

            remaining_samples = preview_rows - len(samples)
            if remaining_samples > 0:
                for row_index in original.index[changed_mask][:remaining_samples]:
                    samples.append(
                        {
                            "data_row": int(row_index) + 1,
                            "before": preview_value(original.at[row_index]),
                            "after": preview_value(normalized.at[row_index]),
                        }
                    )

            chunk[text_column] = normalized
            chunk.to_csv(
                temporary_path,
                mode="a",
                sep=delimiter,
                encoding=output_encoding,
                index=False,
                header=False,
                lineterminator="\n",
            )

            rows_in_chunk = len(chunk)
            total_rows += rows_in_chunk
            changed_rows += int(changed_mask.sum())
            empty_text_rows += int(normalized.eq("").sum())

            if show_progress:
                print(
                    f"Chunk {chunk_count:,}: {rows_in_chunk:,} dòng | "
                    f"Tổng đã xử lý: {total_rows:,} dòng"
                )

        if output_path.exists() and not overwrite_output:
            raise FileExistsError(
                f"File đầu ra xuất hiện trong lúc xử lý: {output_path}"
            )
        os.replace(temporary_path, output_path)
    except Exception:
        temporary_path.unlink(missing_ok=True)
        raise

    elapsed_seconds = time.perf_counter() - started_at
    result = {
        "input_csv": str(input_path),
        "output_csv": str(output_path),
        "text_column": text_column,
        "chunks": chunk_count,
        "total_rows": total_rows,
        "changed_rows": changed_rows,
        "empty_text_rows": empty_text_rows,
        "elapsed_seconds": round(elapsed_seconds, 3),
        "samples": samples,
    }

    if show_progress:
        print("\nHoàn tất tiền xử lý.")
        print(f"File đầu ra: {output_path}")
        print(f"Số chunk: {chunk_count:,}")
        print(f"Tổng số dòng: {total_rows:,}")
        print(f"Số dòng text thay đổi: {changed_rows:,}")
        print(f"Số dòng text trống: {empty_text_rows:,}")
        print(f"Thời gian: {elapsed_seconds:.2f} giây")

    return result

## Kiểm thử nhanh

Cell dưới đây dùng thư mục tạm và CSV nhỏ để kiểm tra hàm chuẩn hóa, xử lý nhiều chunk, bảo toàn schema/thứ tự, giữ mã có số 0 đầu, và các lỗi cấu hình quan trọng. Không tạo file trong dự án.

In [ ]:
def run_self_tests():
    decomposed_vietnamese = unicodedata.normalize("NFD", "Tiếng Việt")
    assert normalize_text(decomposed_vietnamese) == "Tiếng Việt"
    assert normalize_text("  Dòng\t một  \r\n\r\n\r\n Dòng\u00a0hai  "
                         ) == "Dòng một\n\nDòng hai"
    assert normalize_text("") == ""
    assert normalize_text(None) is None
    assert normalize_text(pd.NA) is pd.NA

    with tempfile.TemporaryDirectory() as temporary_directory:
        temporary_directory = Path(temporary_directory)
        source_path = temporary_directory / "source.csv"
        result_path = temporary_directory / "result.csv"

        source = pd.DataFrame(
            {
                "id": ["001", "002", "003", "004", "005"],
                "text": [
                    decomposed_vietnamese,
                    "  nhiều   khoảng\tcách  ",
                    "đoạn một\r\n\r\n\r\nđoạn hai",
                    "",
                    "đã sạch",
                ],
                "metadata": ["A", "B", "C", "D", "E"],
            }
        )
        source.to_csv(source_path, index=False, encoding="utf-8")

        stats = preprocess_csv(
            source_path,
            result_path,
            "text",
            chunk_size=2,
            preview_rows=2,
            show_progress=False,
        )
        result = pd.read_csv(
            result_path,
            dtype=str,
            keep_default_na=False,
            na_filter=False,
        )

        assert stats["chunks"] == 3
        assert stats["total_rows"] == len(source) == len(result)
        assert result.columns.tolist() == source.columns.tolist()
        assert result["id"].tolist() == source["id"].tolist()
        assert result["metadata"].tolist() == source["metadata"].tolist()
        assert result["text"].tolist() == [
            "Tiếng Việt",
            "nhiều khoảng cách",
            "đoạn một\n\nđoạn hai",
            "",
            "đã sạch",
        ]

        try:
            preprocess_csv(source_path, result_path, "text", show_progress=False)
            raise AssertionError("Phải từ chối ghi đè file đầu ra.")
        except FileExistsError:
            pass

        try:
            preprocess_csv(source_path, temporary_directory / "missing-column.csv", "unknown", show_progress=False)
            raise AssertionError("Phải báo lỗi khi thiếu cột text.")
        except KeyError:
            pass

        try:
            preprocess_csv(temporary_directory / "missing.csv", temporary_directory / "unused.csv", "text", show_progress=False)
            raise AssertionError("Phải báo lỗi khi thiếu file nguồn.")
        except FileNotFoundError:
            pass

        malformed_path = temporary_directory / "malformed.csv"
        malformed_path.write_text('id,text\n1,"unclosed\n', encoding="utf-8")
        try:
            preprocess_csv(malformed_path, temporary_directory / "malformed-output.csv", "text", show_progress=False)
            raise AssertionError("Phải báo lỗi khi CSV sai cấu trúc.")
        except pd.errors.ParserError:
            pass

    print("Tất cả kiểm thử đều thành công.")


run_self_tests()

## Chạy tiền xử lý

Sau khi cập nhật ba giá trị trong cell cấu hình, chạy cell này. Bảng cuối cùng chỉ hiển thị tối đa `PREVIEW_ROWS` mẫu đã thay đổi và không ảnh hưởng đến file kết quả.

In [ ]:
preprocessing_result = preprocess_csv(
    INPUT_CSV,
    OUTPUT_CSV,
    TEXT_COLUMN,
    chunk_size=CHUNK_SIZE,
    input_encoding=INPUT_ENCODING,
    output_encoding=OUTPUT_ENCODING,
    delimiter=DELIMITER,
    preview_rows=PREVIEW_ROWS,
    overwrite_output=OVERWRITE_OUTPUT,
)

pd.DataFrame(preprocessing_result["samples"])